# Smart Campus Navigation and Route Optimization System Using Python

**Objective:** Represent campus locations as a weighted graph and find shortest routes.

> Distances are illustrative sample values, not official campus measurements.

## 1. Build the campus graph

In [ ]:
import heapq, sqlite3
from datetime import datetime
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt

EDGES = [('Main Gate', 'Admin Block', 250), ('Main Gate', 'Canteen', 180), ('Admin Block', 'Library', 220), ('Admin Block', 'Academic Block', 160), ('Canteen', 'Library', 270), ('Canteen', 'Hostel', 350), ('Library', 'Academic Block', 120), ('Library', 'Auditorium', 200), ('Academic Block', 'Auditorium', 140), ('Academic Block', 'Hostel', 300), ('Auditorium', 'Sports Ground', 260), ('Hostel', 'Sports Ground', 210)]
G = nx.Graph()
for start, end, distance in EDGES:
    G.add_edge(start, end, weight=distance)
print('Locations:', sorted(G.nodes()))
print('Connections:', G.number_of_edges())

## 2. Dijkstra's shortest-path algorithm

In [ ]:
def dijkstra(graph, source, destination):
    distances = {node: float('inf') for node in graph.nodes}
    previous = {}
    distances[source] = 0
    queue = [(0, source)]
    while queue:
        current_distance, current = heapq.heappop(queue)
        if current_distance != distances[current]:
            continue
        if current == destination:
            break
        for neighbor, data in graph[current].items():
            candidate = current_distance + data['weight']
            if candidate < distances[neighbor]:
                distances[neighbor] = candidate
                previous[neighbor] = current
                heapq.heappush(queue, (candidate, neighbor))
    if distances[destination] == float('inf'):
        return None, float('inf')
    path, current = [], destination
    while current != source:
        path.append(current)
        current = previous[current]
    path.append(source)
    path.reverse()
    return path, distances[destination]

## 3. Try a route

In [ ]:
source, destination = 'Main Gate', 'Library'
route, distance = dijkstra(G, source, destination)
minutes = distance / 80
print('Route:', ' -> '.join(route))
print(f'Distance: {distance:.0f} metres')
print(f'Estimated walking time: {minutes:.1f} minutes')

## 4. Visualize the route

In [ ]:
positions = nx.spring_layout(G, seed=17, weight='weight')
plt.figure(figsize=(12, 7))
nx.draw_networkx_nodes(G, positions, node_size=1800)
nx.draw_networkx_labels(G, positions, font_size=8)
nx.draw_networkx_edges(G, positions, edge_color='gray')
nx.draw_networkx_edge_labels(G, positions, edge_labels=nx.get_edge_attributes(G, 'weight'), font_size=7)
nx.draw_networkx_nodes(G, positions, nodelist=route, node_size=1900, node_color='orange')
nx.draw_networkx_edges(G, positions, edgelist=list(zip(route, route[1:])), edge_color='red', width=4)
plt.title('Smart Campus Navigation — sample distances in metres')
plt.axis('off')
plt.show()

## 5. Find an alternative route if a connection is blocked

In [ ]:
blocked_graph = G.copy()
blocked_graph.remove_edge('Main Gate', 'Canteen')
alternative_route, alternative_distance = dijkstra(blocked_graph, 'Main Gate', 'Library')
print('Alternative route:', ' -> '.join(alternative_route) if alternative_route else 'No route available')
print('Alternative distance:', alternative_distance, 'metres')

## 6. Store a route search in SQLite

In [ ]:
db_path = Path('route_history_notebook.db')
with sqlite3.connect(db_path) as connection:
    connection.execute('''CREATE TABLE IF NOT EXISTS route_history (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        source TEXT NOT NULL, destination TEXT NOT NULL, route TEXT NOT NULL,
        distance_m REAL NOT NULL, walking_minutes REAL NOT NULL, searched_at TEXT NOT NULL
    )''')
    connection.execute('''INSERT INTO route_history
        (source, destination, route, distance_m, walking_minutes, searched_at)
        VALUES (?, ?, ?, ?, ?, ?)''',
        (source, destination, ' -> '.join(route), distance, minutes, datetime.now().isoformat(timespec='seconds')))
    rows = connection.execute('SELECT source, destination, route, distance_m, walking_minutes, searched_at FROM route_history').fetchall()
rows

## 7. Conclusion

This prototype uses a weighted graph, a manually implemented Dijkstra algorithm, Matplotlib visualization, and SQLite history. Replace sample locations and distances with verified campus data before final submission.